# 05 · Producción

Así se usa la estrategia: entrena con todos los meses hasta dos antes del mes a predecir, promedia **20 semillas** (semillerío) y elige los **4.500 clientes** con mayor probabilidad (el medio de la banda).

- **No usa la clase del mes a predecir.**
- Las semillas que ya entrenó el `03` para ese mes son el mismo modelo: se reusan sus predicciones.
- Solo reusa lo guardado si es de la configuración actual (`config.json`).

In [ ]:
import importlib
import time

import numpy as np
import pandas as pd

import warnings

import parametros as P
import funciones as F

warnings.filterwarnings('ignore', message='.*ChainedAssignmentError.*')   # falso positivo de pandas 2.2 con Python 3.14

importlib.reload(P)
importlib.reload(F)
P.RESULTADOS.mkdir(parents=True, exist_ok=True)
print('Corrida:', P.CORRIDA['nombre'], '·', len(P.CORRIDA['bloques']), 'bloques')

In [ ]:
import warnings

warnings.filterwarnings('ignore', message='Cannot add features from NoneType')
c = P.CORRIDA
F.revisar_config(P, c)                               # corta si lo guardado es de otra configuración
mes = P.MES_PRODUCCION
info = F.leer_filas(P)
SEMILLAS = F.semillas(P, P.N_SEMILLAS_PRODUCCION)
meses_train = F.ventana(c, mes, P)
carpeta = P.EXP / c['nombre'] / 'produccion' / str(mes)
carpeta.mkdir(parents=True, exist_ok=True)
print(f'{c["nombre"]}: entrena {meses_train[0]}-{meses_train[-1] % 100:02d} · predice {mes} · {len(SEMILLAS)} semillas')

In [ ]:
tabla = F.Tabla(P, c['bloques'])
es_mes = info['mes_fila'] == mes
falta = [s for s in SEMILLAS if not (carpeta / f'{s}.npy').exists()]
for s in list(falta):
    previa = F.ruta_pred(P, c['nombre'], mes, s)
    if previa.exists():                                # el 03 ya entrenó este modelo
        F.guardar_npy(carpeta / f'{s}.npy', np.load(previa))
        falta.remove(s)
if falta:
    entrena = np.isin(info['mes_fila'], meses_train)
    dset = F.dataset(tabla, entrena, info['clase'][entrena], c['peso_b2'], P)
    X_mes = tabla.filas(es_mes)
    for s in falta:
        t0 = time.time()
        F.guardar_npy(carpeta / f'{s}.npy', F.entrenar(dset, c, s, P).predict(X_mes, num_threads=P.N_HILOS).astype(np.float32))
        print(f'semilla {s}: {time.time() - t0:.0f} s · memoria máxima {F.memoria_gb()} GB', flush=True)
    del dset, X_mes
print('Modelos completos:', len(SEMILLAS))

In [ ]:
prob = np.mean([np.load(carpeta / f'{s}.npy') for s in SEMILLAS], axis=0)
orden = np.argsort(-prob, kind='stable')
envios = pd.DataFrame({'numero_de_cliente': info['cliente'][es_mes][orden[:P.ENVIOS_PRODUCCION]],
                       'probabilidad': prob[orden[:P.ENVIOS_PRODUCCION]]})
destino = P.EXP / c['nombre'] / 'produccion' / f'envios_{mes}.csv'
envios.to_csv(destino, index=False)
print(f'{len(envios):,} envíos en {destino} · probabilidad de corte {envios["probabilidad"].iloc[-1]:.4f}')